In [ ]:
#------------------------------------------------ Begin_Libraries ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import requests
import datetime
import os
import urllib3

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_Setup ----------------------------------------
regulatorName = 'TL BCTL'  # Timor-Leste Banco Central

print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
# writer = ExcelWriter(filename)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running TL BCTL Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_SQLDictionary ----------------------------------------
sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 
    'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [], 
    'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 
    'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 
    'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
    'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 
    'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 
    'Name - Mother Company': [], 'Address_1 - Mother company': [], 'Address_2 - Mother company': [], 
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
    'Phone - Mother company': [], 'Check': []
}

processdate = now.strftime('%Y-%m-%d')

# Region configuration
regdict = {
    'TL BCTL 1': 'https://www.bancocentral.tl/en/go/financial-institution'
}

typology = {
    'TL BCTL 1': 'List of Supervised Financial Institutions'
}

In [4]:
#------------------------------------------------ Begin_Functions ----------------------------------------
def balance_array_lengths(sqldict):
    """Ensure all arrays in sqldict have the same length by padding with empty strings"""
    maxlen = max(len(v) for v in sqldict.values()) if sqldict.values() else 0
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = [''] * (maxlen - len(sqldict[key]))
            sqldict[key] = sqldict[key] + empty
    return sqldict

def extract_table_data(table_element):
    """Extract key-value pairs from a table element"""
    data = {}
    if not table_element:
        return data
    
    rows = table_element.find_all('tr')
    for row in rows:
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 2:
            key = cols[0].get_text(strip=True).upper()
            value = cols[1].get_text(strip=True)
            # Remove HTML tags if any
            if value:
                data[key] = value
    return data

def extract_institution_data(section_div):
    """Extract institution name and its associated table data"""
    institutions = []
    
    # Find all institution rows (documento-cat-row)
    inst_rows = section_div.find_all('div', class_='documento-cat-row')
    
    for inst_row in inst_rows:
        # Get institution name from link
        link = inst_row.find('a', class_='documento-cat-link')
        if not link:
            continue
        
        inst_name = link.get_text(strip=True)
        if not inst_name:
            continue
        
        # Find the associated detail div (next sibling documentos-cat-body)
        detail_div = inst_row.find_next('div', class_='documentos-cat-body')
        table_data = {}
        
        if detail_div:
            # Extract all tables from detail div
            tables = detail_div.find_all('table')
            for table in tables:
                table_data.update(extract_table_data(table))
        
        institutions.append({
            'name': inst_name,
            'details': table_data
        })
    
    return institutions

In [ ]:
#------------------------------------------------ Begin_Main_Scraping ----------------------------------------
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Map category titles to list codes and typologies
category_mapping = {
    'Commercial Banks': {'ListCode': 'CB', 'Typology': 'Commercial Bank'},
    'MONEY TRANSFER OPERATORS': {'ListCode': 'MTO', 'Typology': 'Money Transfer Operator'},
    'CURRENCY EXCHANGE BURREAUX (CEB)': {'ListCode': 'CEB', 'Typology': 'Currency Exchange Bureau'},
    'Insurance Companies': {'ListCode': 'IC', 'Typology': 'Insurance Company'},
    'Other Deposit Taking Institutions (ODTIs)': {'ListCode': 'ODTI', 'Typology': 'Other Deposit Taking Institution'},
    'Finance Company': {'ListCode': 'FC', 'Typology': 'Finance Company'},
    'FINTECH COMPANIES': {'ListCode': 'FINTECH', 'Typology': 'FinTech Company'}
}

# Headers for request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

url = 'https://www.bancocentral.tl/en/go/financial-institution'

print(f"[INFO] : Fetching {url}")

try:
    res = requests.get(url, headers=headers, verify=False, timeout=30)
    res.raise_for_status()
    
    # Parse HTML
    soup = BeautifulSoup(res.text, 'html.parser')
    
    # Find main content div
    main_div = soup.find('div', class_='col-sm-12 rem-padding documentos-list faqs')
    
    if not main_div:
        print("[ERROR] : Could not find main content div")
    else:
        print("[INFO] : Found main content div")
        
        # Split by category
        category_divs = main_div.find_all('div', class_='page-title documentos')
        
        for cat_idx, cat_title_div in enumerate(category_divs):
            # Get category name
            category_text = cat_title_div.get_text(strip=True)
            
            # Find the next documentos-body div which contains the institutions
            doc_body = cat_title_div.find_next('div', class_='documentos-body')
            
            if not doc_body:
                continue
            
            # Get mapping info
            cat_info = category_mapping.get(category_text, 
                                           {'ListCode': 'UNK', 'Typology': category_text})
            list_code = cat_info['ListCode']
            typology = cat_info['Typology']
            
            print(f"\n[INFO] : Processing category: {category_text}")
            
            # Extract institutions in this category
            institutions = extract_institution_data(doc_body)
            
            print(f"[INFO] : Found {len(institutions)} institutions in {category_text}")
            
            # Add to sqldict
            for inst in institutions:
                inst_name = inst['name']
                details = inst['details']
                
                sqldict['Name'].append(inst_name)
                sqldict['Typology'].append(typology)
                sqldict['RegulationType'].append('Regulated')
                sqldict['Cntry'].append('TL')
                sqldict['RegCtry'].append('TL')
                sqldict['RegCode'].append('BCTL')
                sqldict['ListCode'].append('1')
                sqldict['ListName'].append('Supervised Financial Institutions')
                sqldict['ListProcessDate'].append(processdate)
                
                
                # Extract mapped fields from table data
                address_1 = details.get('STREET ADDRESS', details.get('ADDRESS', ''))
                sqldict['Address_1'].append(address_1 if len(address_1) >= 3 else '')
                
                phone = details.get('TELEPHONE', details.get('CALL CENTER NUMBER', 
                                    details.get('MOBILE NUMBER', '')))
                sqldict['Phone'].append(phone if len(phone) >= 3 else '')
                
                fax = details.get('TELEFAX', details.get('FACSIMILE', ''))
                sqldict['Fax'].append(fax if len(fax) >= 3 else '')
                
                email = details.get('EMAIL', '')
                sqldict['Email'].append(email if len(email) >= 3 else '')
                
                website = details.get('HOME PAGE', details.get('WEBSITE', ''))
                sqldict['Website'].append(website if len(website) >= 3 else '')
                
                license_no = details.get('LICENSE NUMBER', details.get('LICENSE Nº', 
                                        details.get('LICENCE Nº', '')))
                sqldict['InternalID_1'].append(license_no)
                sqldict['InternalID_1_type'].append('License Number')
                
                swift_code = details.get('SWIFT CODE', '')
                sqldict['BIC SWIFT Code'].append(swift_code)
                
                # Fill remaining fields with empty strings
                sqldict = balance_array_lengths(sqldict)
    
    # Balance array lengths
    sqldict = balance_array_lengths(sqldict)
    
    print(f"\n[INFO] : Total records extracted: {len(sqldict['Name'])}")
    
except Exception as e:
    print(f"[ERROR] : {str(e)}")

print("\n[INFO] : Scraping completed")

[INFO] : Fetching https://www.bancocentral.tl/en/go/financial-institution
[INFO] : Found main content div

[INFO] : Processing category: Commercial Banks
[INFO] : Found 6 institutions in Commercial Banks

[INFO] : Processing category: MONEY TRANSFER OPERATORS
[INFO] : Found 10 institutions in MONEY TRANSFER OPERATORS

[INFO] : Processing category: CURRENCY EXCHANGE BURREAUX (CEB)
[INFO] : Found 4 institutions in CURRENCY EXCHANGE BURREAUX (CEB)

[INFO] : Processing category: Insurance Companies
[INFO] : Found 3 institutions in Insurance Companies

[INFO] : Processing category: Other Deposit Taking Institutions (ODTIs)
[INFO] : Found 2 institutions in Other Deposit Taking Institutions (ODTIs)

[INFO] : Processing category: Finance Company
[INFO] : Found 1 institutions in Finance Company

[INFO] : Processing category: FINTECH COMPANIES
[INFO] : Found 3 institutions in FINTECH COMPANIES

[INFO] : Total records extracted: 29

[INFO] : Scraping completed


In [ ]:
#------------------------------------------------ Begin_Save_Excel ----------------------------------------
# Balance final array lengths
sqldict = balance_array_lengths(sqldict)

# Convert to DataFrame
df = pd.DataFrame(sqldict)

print(f"\n[INFO] : Final DataFrame shape: {df.shape}")
print(f"[INFO] : Total columns: {len(df.columns)}")

# Save to Excel
os.chdir(scriptfolder)
df.to_excel(filename, 'SQL Ready', index=False)

print(f"[INFO] : File saved: {filename}")

# Cleanup temp folder
for rem in os.listdir(tempfolder):
    try:
        os.remove(os.path.join(tempfolder, rem))
    except:
        pass

print("[INFO] : Scraping process completed successfully!")


[INFO] : Final DataFrame shape: (29, 44)
[INFO] : Total columns: 44
[INFO] : File saved: TL BCTL SQL Ready 2026-04-14 17.31.13.xlsx
[INFO] : Excel writer closed
[INFO] : Scraping process completed successfully!


In [10]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,Commercial Banks,Commercial Bank,Regulated Entity,BNU TIMOR – GRUPO CAIXA GERAL DE DEPÓSITOS,CPO/01/2000,License Number,,,...,,CGDITLDI,,,,,,,,
1,,,Commercial Banks,Commercial Bank,Regulated Entity,AUSTRALIA AND NEW ZEALAND BANK (ANZ) BANKING G...,CPO/B/02/2001,License Number,,,...,,ANZBTLDI,,,,,,,,
2,,,Commercial Banks,Commercial Bank,Regulated Entity,BANCO NACIONAL DE COMÉRCIO DE TIMOR-LESTE (BNCTL),BCTL/169/2026,License Number,,,...,,BNFTTLDI,,,,,,,,
3,,,Commercial Banks,Commercial Bank,Regulated Entity,PT. BANK MANDIRI (PERSERO) TBK. DILI - TIMOR-L...,BPA/B/04/2003,License Number,,,...,,BEIIIDJAXXX,,,,,,,,
4,,,Commercial Banks,Commercial Bank,Regulated Entity,"PT. BANK RAKYAT INDONESIA (PERSERO), TBK, TIMO...",BCTL/B/05/2017,License Number,,,...,,BRINTLDI,,,,,,,,
5,,,Commercial Banks,Commercial Bank,Regulated Entity,"BANCO DO NOSSO FUTURO, S.A",BCTL/156/2025,License Number,,,...,,BNFTTLDI,,,,,,,,
6,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"SAHABAT LORO SA'E, UNIPESSOAL, LDA ""MONEY TRAN...",GD No.4/2013,License Number,,,...,,,,,,,,,,
7,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"ISLAND DREAM MONEY, LDA ""MONEY TRANSFER""",GD No. 5/2013,License Number,,,...,,,,,,,,,,
8,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"SISTER MOTOR II UNIPESSOAL, LDA ""MONEY TRANSFE...",GD No. 6/2013,License Number,,,...,,,,,,,,,,
9,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"BOA VENTURA UNIPESSOAL, LDA "" MONEY TRANSFER O...",GD No. 8/2013,License Number,,,...,,,,,,,,,,


In [9]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,Commercial Banks,Commercial Bank,Regulated Entity,BNU TIMOR – GRUPO CAIXA GERAL DE DEPÓSITOS,CPO/01/2000,License Number,,,...,,CGDITLDI,,,,,,,,
1,,,Commercial Banks,Commercial Bank,Regulated Entity,AUSTRALIA AND NEW ZEALAND BANK (ANZ) BANKING G...,CPO/B/02/2001,License Number,,,...,,ANZBTLDI,,,,,,,,
2,,,Commercial Banks,Commercial Bank,Regulated Entity,BANCO NACIONAL DE COMÉRCIO DE TIMOR-LESTE (BNCTL),BCTL/169/2026,License Number,,,...,,BNFTTLDI,,,,,,,,
3,,,Commercial Banks,Commercial Bank,Regulated Entity,PT. BANK MANDIRI (PERSERO) TBK. DILI - TIMOR-L...,BPA/B/04/2003,License Number,,,...,,BEIIIDJAXXX,,,,,,,,
4,,,Commercial Banks,Commercial Bank,Regulated Entity,"PT. BANK RAKYAT INDONESIA (PERSERO), TBK, TIMO...",BCTL/B/05/2017,License Number,,,...,,BRINTLDI,,,,,,,,
5,,,Commercial Banks,Commercial Bank,Regulated Entity,"BANCO DO NOSSO FUTURO, S.A",BCTL/156/2025,License Number,,,...,,BNFTTLDI,,,,,,,,
6,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"SAHABAT LORO SA'E, UNIPESSOAL, LDA ""MONEY TRAN...",GD No.4/2013,License Number,,,...,,,,,,,,,,
7,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"ISLAND DREAM MONEY, LDA ""MONEY TRANSFER""",GD No. 5/2013,License Number,,,...,,,,,,,,,,
8,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"SISTER MOTOR II UNIPESSOAL, LDA ""MONEY TRANSFE...",GD No. 6/2013,License Number,,,...,,,,,,,,,,
9,,,MONEY TRANSFER OPERATORS,Money Transfer Operator,Regulated Entity,"BOA VENTURA UNIPESSOAL, LDA "" MONEY TRANSFER O...",GD No. 8/2013,License Number,,,...,,,,,,,,,,
